# Phase 3 — Preprocessing Pipeline

**Course**: CS5143 Natural Language Processing — Spring 2026, FAST-NUCES  
**Student**: Muhammad Azhar (24K-7606)

This notebook tokenizes the raw CoNLL data using the XLM-R SentencePiece tokenizer,
aligns BIO labels to subword tokens, and saves HuggingFace `DatasetDict` objects to disk
for use in Phase 4 (training).

**Prerequisites**: Complete Phase 2 — `data/raw/en/` and `data/raw/es/` must contain the
CoNLL files (`train.conll`, `dev.conll`, `test.conll`).

## 1. Setup

In [1]:
import sys
import os
from pathlib import Path

# Make src/ importable regardless of where the notebook is launched
REPO_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
sys.path.insert(0, str(REPO_ROOT / 'src'))

from utils import (
    tokenizer,
    build_hf_dataset,
    build_joint_dataset,
    get_data_paths,
    MODEL_NAME,
    MAX_LENGTH,
)
from dataset import id2label, LABEL_LIST

DATA_RAW  = REPO_ROOT / 'data' / 'raw'
DATA_PROC = REPO_ROOT / 'data' / 'processed'

print(f'Model        : {MODEL_NAME}')
print(f'Max length   : {MAX_LENGTH}')
print(f'Labels ({len(LABEL_LIST)}): {LABEL_LIST}')
print(f'Tokenizer    : {tokenizer.__class__.__name__}')

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Model        : xlm-roberta-base
Max length   : 128
Labels (7): ['O', 'B-DIS', 'I-DIS', 'B-SYM', 'I-SYM', 'B-PRO', 'I-PRO']
Tokenizer    : XLMRobertaTokenizer


## 2. Subword Alignment — Key Concept

XLM-R uses SentencePiece tokenization: one word may split into multiple subword tokens.
We use the **first-subword strategy**:
- Assign the original BIO label to the **first subword** of each word
- Assign `-100` to all continuation subwords and to special tokens `[CLS]`/`[SEP]`

PyTorch's `CrossEntropyLoss` ignores positions with label `-100`, so continuations do not
contribute to the loss.

In [2]:
# Demonstrate subword splitting on clinical terms
demo_words = ['hypertension', 'dyspnea', 'laparoscopy', 'pneumonia', 'tachycardia']

print(f'{'Word':<20} {'Subwords'}')
print('-' * 60)
for word in demo_words:
    subwords = tokenizer.tokenize(word)
    print(f'{word:<20} {subwords}')

Word                 Subwords
------------------------------------------------------------
hypertension         ['▁hyper', 'tension']
dyspnea              ['▁dys', 'pne', 'a']
laparoscopy          ['▁la', 'paro', 's', 'copy']
pneumonia            ['▁pneu', 'monia']
tachycardia          ['▁ta', 'chy', 'car', 'dia']


## 3. Build HuggingFace Datasets

We build separate datasets for English and Spanish, then a joint dataset that concatenates
both training sets. Joint training enables cross-lingual transfer between languages.

In [3]:
print('Building English dataset ...')
en_train_f, en_dev_f, en_test_f = get_data_paths('en')
en_ds = build_hf_dataset(en_train_f, en_dev_f, en_test_f)
print(en_ds)

print('\nBuilding Spanish dataset ...')
es_train_f, es_dev_f, es_test_f = get_data_paths('es')
es_ds = build_hf_dataset(es_train_f, es_dev_f, es_test_f)
print(es_ds)

print('\nBuilding joint EN+ES dataset ...')
joint_ds = build_joint_dataset(en_ds, es_ds)
print(joint_ds)
print(f'\nJoint train size: {len(joint_ds["train"])} '
      f'(EN {len(en_ds["train"])} + ES {len(es_ds["train"])})')

Building English dataset ...


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2315 [00:00<?, ? examples/s]

Map:   0%|          | 0/2250 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2315
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2250
    })
})

Building Spanish dataset ...


Map:   0%|          | 0/10163 [00:00<?, ? examples/s]

Map:   0%|          | 0/2151 [00:00<?, ? examples/s]

Map:   0%|          | 0/2251 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10163
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2151
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2251
    })
})

Building joint EN+ES dataset ...
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 20163
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2315
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2250
    })
})

Joint train size: 20163 (EN 10000 + ES 10163)


## 4. Verify Subword Label Alignment

Inspect one tokenized sentence to confirm that:
- First subwords carry the correct BIO label
- Continuation subwords show `(-100)`
- `[CLS]` and `[EOS]` tokens show `(-100)`

In [4]:
# Pick the first sentence from the English training set that has at least one entity
sample = None
for item in en_ds['train']:
    if any(l != -100 and id2label[l] != 'O' for l in item['labels']):
        sample = item
        break

if sample is None:
    sample = en_ds['train'][0]

tokens = tokenizer.convert_ids_to_tokens(sample['input_ids'])
labels = sample['labels']

print(f'{'Subword token':<25} {'Label'}')
print('-' * 40)
for tok, lbl in zip(tokens, labels):
    lbl_str = id2label[lbl] if lbl != -100 else '(-100)'
    marker  = ' ◄' if lbl != -100 and id2label[lbl] != 'O' else ''
    print(f'{tok:<25} {lbl_str}{marker}')

Subword token             Label
----------------------------------------
<s>                       (-100)
▁A                        O
▁42                       O
-                         (-100)
year                      (-100)
-                         (-100)
old                       (-100)
▁woman                    O
▁consulte                 O
d                         (-100)
▁for                      O
▁a                        O
▁history                  O
▁of                       O
▁head                     O
ache                      (-100)
▁of                       O
▁10                       O
▁days                     O
'                         (-100)
▁du                       O
ration                    (-100)
▁ac                       O
com                       (-100)
pani                      (-100)
ed                        (-100)
▁by                       O
▁ga                       O
it                        (-100)
▁in                       O
st                    

## 5. Save Processed Datasets to Disk

Saving the tokenized datasets avoids re-running tokenization before every training run.
The training script (`src/train.py`) loads from these paths via `load_from_disk()`.

In [5]:
DATA_PROC.mkdir(parents=True, exist_ok=True)

en_ds.save_to_disk(str(DATA_PROC / 'en'))
print(f'Saved EN dataset    → {DATA_PROC / "en"}')

es_ds.save_to_disk(str(DATA_PROC / 'es'))
print(f'Saved ES dataset    → {DATA_PROC / "es"}')

joint_ds.save_to_disk(str(DATA_PROC / 'joint'))
print(f'Saved joint dataset → {DATA_PROC / "joint"}')

print('\nAll processed datasets saved. Phase 3 complete.')

Saving the dataset (0/1 shards):   0%|          | 0/10000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2315 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2250 [00:00<?, ? examples/s]

Saved EN dataset    → /home/azharthegeek/Desktop/work/MultiClinNER-XLMR/data/processed/en


Saving the dataset (0/1 shards):   0%|          | 0/10163 [00:00<?, ? examples/s]

...

Saved ES dataset    → /home/azharthegeek/Desktop/work/MultiClinNER-XLMR/data/processed/es


Saving the dataset (0/1 shards):   0%|          | 0/20163 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2315 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2250 [00:00<?, ? examples/s]

Saved joint dataset → /home/azharthegeek/Desktop/work/MultiClinNER-XLMR/data/processed/joint

All processed datasets saved. Phase 3 complete.


## Summary

**What was done in this phase:**
- Loaded raw CoNLL files for English and Spanish
- Applied XLM-R SentencePiece tokenization with first-subword label alignment
- Built three HuggingFace DatasetDicts: `en`, `es`, and `joint`
- Saved tokenized datasets to `data/processed/` for fast re-use

**Next step**: Phase 4 — Model Training (`notebooks/03_model_training.ipynb`)